In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import cftime
import os

In [2]:
llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch_full = xr.open_dataset(llc_path, consolidated=True)

# EXPERIMENT 1
#epochs=1, steps=1, vars=all, loss=mse_diff_weighted, norm=syncbatchnorm, padding='constant,' pred_residual=true,
emulator_1_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_1/predictions_4d.zarr'
emulator_1_patch_full = xr.open_dataset(emulator_1_path, consolidated=True) 

# EXPERIMENT 2
# epochs=1, steps=1, vars=all, loss=mse_diff_weighted, norm=group_norm_32, padding='constant,' pred_residual=true,
emulator_2_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_2/predictions_4d.zarr'
emulator_2_patch_full = xr.open_dataset(emulator_2_path, consolidated=True) 

# EXPERIMENT 3
# epochs=1, steps=1, vars=all, loss=mse_diff_weighted + dynamically weighted loss, norm=syncbatchnorm, padding='constant,' pred_residual=true,
# emulator_3_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_3/predictions_4d.zarr'
# emulator_3_patch_full = xr.open_dataset(emulator_3_path, consolidated=True) 

# EXPERIMENT 4
# epochs=1, steps=1, vars=all, loss=mse_diff_weighted + dynamically weighted loss, norm=syncbatchnorm, padding='halo_sponge,' pred_residual=true
#emulator_4_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_4/predictions_4d.zarr'
#emulator_4_patch_full = xr.open_dataset(emulator_4_path, consolidated=True) 

# EXPERIMENT 5
# epochs=1, steps=1, vars=all, loss=mse_diff_weighted + weighted loss [U,V = 1.0, Theta, Salt, Eta = 1.5 , norm=group_norm_32, padding='constant,' pred_residual=true,
emulator_3_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_5/predictions_4d.zarr'
emulator_3_patch_full = xr.open_dataset(emulator_3_path, consolidated=True) 

# EXPERIMENT 6
# epochs=1, steps=1, vars=all, loss=mae gradient, norm=group_norm_32, padding='constant,' pred_residual=true,
emulator_4_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_6/predictions_4d.zarr'
emulator_4_patch_full = xr.open_dataset(emulator_4_path, consolidated=True) 

In [3]:
emulator_times = emulator_1_patch_full.time.values
matching_times = pd.DatetimeIndex([
    pd.Timestamp(t.year, t.month, t.day, t.hour, t.minute, t.second)
    for t in emulator_times
])

llc_patch = llc_patch_full.sel(time=matching_times)

In [4]:

# emulator_1_patch = emulator_1_patch_full.isel(time=slice(0, extent))
# emulator_2_patch = emulator_2_patch_full.isel(time=slice(0, extent))
# emulator_3_patch = emulator_3_patch_full.isel(time=slice(0, extent))
# emulator_4_patch = emulator_4_patch_full.isel(time=slice(0, extent))

emulator_1_patch = emulator_1_patch_full
emulator_2_patch = emulator_2_patch_full
emulator_3_patch = emulator_3_patch_full
emulator_4_patch = emulator_4_patch_full

In [5]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        # cftime objects
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        # numpy datetime64
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"


# surface field and field difference

In [6]:
# ============== SET VARIABLES HERE ==============
vars = ['Theta', 'Salt', 'U', 'V']
colormaps = {'Theta': 'Spectral_r', 'Salt': 'viridis', 'U': 'bwr', 'V': 'bwr'}
# ================================================

for var in vars:
    print(f"Generating plots for {var}...")
    
    os.makedirs(f'figs/prognostic_var_comparison/{var}', exist_ok=True)
    
    n_times = len(emulator_1_patch.time)
    time_step = 1
    time_indices = list(range(0, n_times, time_step))
    nrows = len(time_indices)
    ncols = 5
    
    cmap = colormaps[var]
    
    # ==================== PLOT 1: Surface fields ====================
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(emulator_1_patch.time.values[t])
        
        llc_vis = llc_patch.isel(time=t, k=0)[var]
        emulator_1_vis = emulator_1_patch.isel(time=t, k=0)[var]
        emulator_2_vis = emulator_2_patch.isel(time=t, k=0)[var]
        emulator_3_vis = emulator_3_patch.isel(time=t, k=0)[var]
        emulator_4_vis = emulator_4_patch.isel(time=t, k=0)[var]
        
        vmin = np.min([llc_vis.values.min(), emulator_1_vis.values.min(), 
                       emulator_2_vis.values.min(), emulator_3_vis.values.min(),
                       emulator_4_vis.values.min()])
        vmax = np.max([llc_vis.values.max(), emulator_1_vis.values.max(), 
                       emulator_2_vis.values.max(), emulator_3_vis.values.max(),
                       emulator_4_vis.values.max()])
        
        ax1, ax2, ax3, ax4, ax5 = axes[row, 0], axes[row, 1], axes[row, 2], axes[row, 3], axes[row, 4]
        
        cf1 = ax1.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), llc_vis, 
                           cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
        ax1.set_title(f'LLC {var} {time_str}', fontsize=8)
        plt.colorbar(cf1, ax=ax1)
        
        cf2 = ax2.contourf(emulator_1_vis.coords.get('i', np.arange(emulator_1_vis.shape[-1])), 
                           emulator_1_vis.coords.get('j', np.arange(emulator_1_vis.shape[-2])), emulator_1_vis, 
                           cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
        ax2.set_title(f'Emulator 1 {var} {time_str}', fontsize=8)
        plt.colorbar(cf2, ax=ax2)
        
        cf3 = ax3.contourf(emulator_2_vis.coords.get('i', np.arange(emulator_2_vis.shape[-1])), 
                           emulator_2_vis.coords.get('j', np.arange(emulator_2_vis.shape[-2])), emulator_2_vis, 
                           cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
        ax3.set_title(f'Emulator 2 {var} {time_str}', fontsize=8)
        plt.colorbar(cf3, ax=ax3)
        
        cf4 = ax4.contourf(emulator_3_vis.coords.get('i', np.arange(emulator_3_vis.shape[-1])), 
                           emulator_3_vis.coords.get('j', np.arange(emulator_3_vis.shape[-2])), emulator_3_vis, 
                           cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
        ax4.set_title(f'Emulator 3 {var} {time_str}', fontsize=8)
        plt.colorbar(cf4, ax=ax4)
        
        cf5 = ax5.contourf(emulator_4_vis.coords.get('i', np.arange(emulator_4_vis.shape[-1])), 
                           emulator_4_vis.coords.get('j', np.arange(emulator_4_vis.shape[-2])), emulator_4_vis, 
                           cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
        ax5.set_title(f'Emulator 4 {var} {time_str}', fontsize=8)
        plt.colorbar(cf5, ax=ax5)
    
    plt.tight_layout()
    plt.savefig(f'figs/prognostic_var_comparison/{var}/surface_{var}_fields.png')
    plt.close()
    
    # ==================== PLOT 2: Difference fields ====================
    fig, axes = plt.subplots(nrows, ncols-1, figsize=(16, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(emulator_1_patch.time.values[t])
        
        llc_vis = llc_patch.isel(time=t, k=0)[var]
        emulator_1_vis = emulator_1_patch.isel(time=t, k=0)[var]
        emulator_2_vis = emulator_2_patch.isel(time=t, k=0)[var]
        emulator_3_vis = emulator_3_patch.isel(time=t, k=0)[var]
        emulator_4_vis = emulator_4_patch.isel(time=t, k=0)[var]
        
        diff_1 = llc_vis.values - emulator_1_vis.values
        diff_2 = llc_vis.values - emulator_2_vis.values
        diff_3 = llc_vis.values - emulator_3_vis.values
        diff_4 = llc_vis.values - emulator_4_vis.values
        
        abs_max = np.max([np.abs(diff_1).max(), np.abs(diff_2).max(), 
                          np.abs(diff_3).max(), np.abs(diff_4).max()])
        vmin, vmax = -abs_max, abs_max
        
        ax1, ax2, ax3, ax4 = axes[row, 0], axes[row, 1], axes[row, 2], axes[row, 3]
        
        cf1 = ax1.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_1, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax1.set_title(f'LLC - Em1 {var} {time_str}', fontsize=8)
        
        cf2 = ax2.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_2, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax2.set_title(f'LLC - Em2 {var} {time_str}', fontsize=8)
        
        cf3 = ax3.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_3, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax3.set_title(f'LLC - Em3 {var} {time_str}', fontsize=8)
        
        cf4 = ax4.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_4, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax4.set_title(f'LLC - Em4 {var} {time_str}', fontsize=8)
        
        fig.colorbar(cf4, ax=[ax1, ax2, ax3, ax4], orientation='vertical', 
                     fraction=0.046, pad=0.04)
    
    plt.savefig(f'figs/prognostic_var_comparison/{var}/surface_{var}_differences.png')
    plt.close()
    
    print(f"Saved plots for {var}")

Generating plots for Theta...
Saved plots for Theta
Generating plots for Salt...
Saved plots for Salt
Generating plots for U...
Saved plots for U
Generating plots for V...
Saved plots for V


# Gradients

In [7]:
emulator_1_patch['XC'] = llc_patch['XC']
emulator_1_patch['YC'] = llc_patch['YC']
emulator_1_patch['rA'] = llc_patch['rA']
emulator_1_patch['Z'] = llc_patch['Z']

emulator_2_patch['XC'] = llc_patch['XC']
emulator_2_patch['YC'] = llc_patch['YC']
emulator_2_patch['rA'] = llc_patch['rA']
emulator_2_patch['Z'] = llc_patch['Z']

emulator_3_patch['XC'] = llc_patch['XC']
emulator_3_patch['YC'] = llc_patch['YC']
emulator_3_patch['rA'] = llc_patch['rA']
emulator_3_patch['Z'] = llc_patch['Z']

emulator_4_patch['XC'] = llc_patch['XC']
emulator_4_patch['YC'] = llc_patch['YC']
emulator_4_patch['rA'] = llc_patch['rA']
emulator_4_patch['Z'] = llc_patch['Z']

In [8]:
vars = ['Theta', 'Salt', 'U', 'V']

all_patches = {
    'llc': llc_patch,
    'emulator_1': emulator_1_patch,
    'emulator_2': emulator_2_patch,
    'emulator_3': emulator_3_patch,
    'emulator_4': emulator_4_patch
}

for var in vars:
    grad_name = f'grad_{var}'
    print(f"Computing {grad_name}...")
    
    for patch_name, patch in all_patches.items():
        data = patch[var].values  # (time, k, j, i)
        
        dx = np.sqrt(patch['rA'].values)  # (j, i) in meters
        dy = dx.copy()
        
        d_di = (np.roll(data, -1, axis=3) - np.roll(data, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
        d_dj = (np.roll(data, -1, axis=2) - np.roll(data, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
        
        grad_mag = np.sqrt(d_di**2 + d_dj**2)
        
        patch[grad_name] = (('time', 'k', 'j', 'i'), grad_mag)
        print(f"  ✓ {patch_name} {grad_name}: {grad_mag.shape}")

print("Done computing gradients!")

Computing grad_Theta...
  ✓ llc grad_Theta: (10, 51, 720, 720)
  ✓ emulator_1 grad_Theta: (10, 51, 720, 720)
  ✓ emulator_2 grad_Theta: (10, 51, 720, 720)
  ✓ emulator_3 grad_Theta: (10, 51, 720, 720)
  ✓ emulator_4 grad_Theta: (10, 51, 720, 720)
Computing grad_Salt...
  ✓ llc grad_Salt: (10, 51, 720, 720)
  ✓ emulator_1 grad_Salt: (10, 51, 720, 720)
  ✓ emulator_2 grad_Salt: (10, 51, 720, 720)
  ✓ emulator_3 grad_Salt: (10, 51, 720, 720)
  ✓ emulator_4 grad_Salt: (10, 51, 720, 720)
Computing grad_U...
  ✓ llc grad_U: (10, 51, 720, 720)
  ✓ emulator_1 grad_U: (10, 51, 720, 720)
  ✓ emulator_2 grad_U: (10, 51, 720, 720)
  ✓ emulator_3 grad_U: (10, 51, 720, 720)
  ✓ emulator_4 grad_U: (10, 51, 720, 720)
Computing grad_V...
  ✓ llc grad_V: (10, 51, 720, 720)
  ✓ emulator_1 grad_V: (10, 51, 720, 720)
  ✓ emulator_2 grad_V: (10, 51, 720, 720)
  ✓ emulator_3 grad_V: (10, 51, 720, 720)
  ✓ emulator_4 grad_V: (10, 51, 720, 720)
Done computing gradients!


In [9]:
# Masks stored as: masks[patch_name][var] -> boolean array (time, k, j, i)
gradient_masks = {}

for patch_name, patch in all_patches.items():
    gradient_masks[patch_name] = {}
    
    for var in vars:
        grad_name = f'grad_{var}'
        grad_data = patch[grad_name].values  # (time, k, j, i)
        
        n_times, n_depths = grad_data.shape[0], grad_data.shape[1]
        mask = np.zeros_like(grad_data, dtype=bool)
        
        for t in range(n_times):
            for k in range(n_depths):
                field = grad_data[t, k]
                # Use nanpercentile to handle NaNs
                threshold = np.nanpercentile(field, 97.5)
                mask[t, k] = field >= threshold
        
        gradient_masks[patch_name][var] = mask
        print(f"✓ {patch_name} {var}: {mask.sum()} high-gradient pixels ({mask.sum() / mask.size * 100:.1f}%)")

print("Done creating gradient masks!")

✓ llc Theta: 6609273 high-gradient pixels (2.5%)
✓ llc Salt: 6609268 high-gradient pixels (2.5%)
✓ llc U: 6609157 high-gradient pixels (2.5%)
✓ llc V: 6609224 high-gradient pixels (2.5%)
✓ emulator_1 Theta: 6609634 high-gradient pixels (2.5%)
✓ emulator_1 Salt: 6609637 high-gradient pixels (2.5%)
✓ emulator_1 U: 6609630 high-gradient pixels (2.5%)
✓ emulator_1 V: 6609642 high-gradient pixels (2.5%)
✓ emulator_2 Theta: 6609638 high-gradient pixels (2.5%)
✓ emulator_2 Salt: 6609642 high-gradient pixels (2.5%)
✓ emulator_2 U: 6609633 high-gradient pixels (2.5%)
✓ emulator_2 V: 6609649 high-gradient pixels (2.5%)
✓ emulator_3 Theta: 6609633 high-gradient pixels (2.5%)
✓ emulator_3 Salt: 6609643 high-gradient pixels (2.5%)
✓ emulator_3 U: 6609623 high-gradient pixels (2.5%)
✓ emulator_3 V: 6609640 high-gradient pixels (2.5%)
✓ emulator_4 Theta: 6609641 high-gradient pixels (2.5%)
✓ emulator_4 Salt: 6609631 high-gradient pixels (2.5%)
✓ emulator_4 U: 6609642 high-gradient pixels (2.5%)
✓ emu

In [10]:
emulator_info = [
    ('Emulator 1', 'emulator_1'),
    ('Emulator 2', 'emulator_2'),
    ('Emulator 3', 'emulator_3'),
    ('Emulator 4', 'emulator_4')
]

for var in vars:
    print(f"Generating gradient drift figure for {var}...")
    
    os.makedirs(f'figs/gradients/{var}', exist_ok=True)
    
    n_times = len(emulator_1_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols = 4
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5*nrows), dpi=150)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(emulator_1_patch.time.values[t])
        
        llc_mask_surface = gradient_masks['llc'][var][t, 0]  # (j, i)
        
        for col, (emulator_name, emulator_key) in enumerate(emulator_info):
            ax = axes[row, col]
            
            # Get emulator surface field
            emu_field = all_patches[emulator_key].isel(time=t, k=0)[var].values  # (j, i)
            emu_mask_surface = gradient_masks[emulator_key][var][t, 0]  # (j, i)
            
            # Plot muted surface field
            ax.imshow(emu_field, cmap='Greys', aspect='auto', origin='lower')
            
            # Overlap mask: both LLC and emulator high-gradient
            overlap_mask = llc_mask_surface & emu_mask_surface
            # LLC only
            llc_only_mask = llc_mask_surface & ~emu_mask_surface
            # Emulator only
            emu_only_mask = emu_mask_surface & ~llc_mask_surface
            
            # Get pixel coordinates
            llc_j, llc_i = np.where(llc_only_mask)
            emu_j, emu_i = np.where(emu_only_mask)
            ovl_j, ovl_i = np.where(overlap_mask)
            
            # Plot: LLC-only in red, emulator-only in blue, overlap in purple
            ax.scatter(llc_i, llc_j, c='red', s=1, alpha=0.5, label='LLC top 2.5%', rasterized=True)
            ax.scatter(emu_i, emu_j, c='green', s=1, alpha=0.5, label='Emu top 2.5%', rasterized=True)
            ax.scatter(ovl_i, ovl_j, c='yellow', s=1, alpha=0.7, label='Overlap', rasterized=True)
            
            # Count overlap pixels
            n_overlap = np.sum(overlap_mask)
            
            ax.set_title(f'{emulator_name} {var} {time_str} overlap={n_overlap}', fontsize=8)
            ax.tick_params(labelsize=6)
            
            if row == 0 and col == 0:
                ax.legend(fontsize=5, loc='upper right', markerscale=5)
    
    plt.tight_layout()
    plt.savefig(f'figs/gradients/{var}/surface_gradient_drift.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved gradient drift figure for {var}")

print("Done with gradient drift figures!")

Generating gradient drift figure for Theta...
✓ Saved gradient drift figure for Theta
Generating gradient drift figure for Salt...
✓ Saved gradient drift figure for Salt
Generating gradient drift figure for U...
✓ Saved gradient drift figure for U
Generating gradient drift figure for V...
✓ Saved gradient drift figure for V
Done with gradient drift figures!


# Error vs depth plots

In [11]:
vars = ['Theta', 'Salt', 'U', 'V']
ref_lines = {
    'Theta': [0.5, 1.0],
    'Salt': [0.06, 0.12],
    'U': [0.075, 0.15],
    'V': [0.075, 0.15]
}

for var in vars:
    print(f"Generating augmented depth error plots for {var}...")
    
    os.makedirs(f'figs/prognostic_var_comparison/{var}', exist_ok=True)
    
    n_depths = llc_patch.sizes['k']
    n_times = len(emulator_1_patch.time)
    time_indices = list(range(n_times))
    
    nrows = len(time_indices)
    ncols = 4
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3*nrows), dpi=150)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    emulator_patches_list = [
        ('Emulator 1', emulator_1_patch, 'emulator_1'),
        ('Emulator 2', emulator_2_patch, 'emulator_2'),
        ('Emulator 3', emulator_3_patch, 'emulator_3'),
        ('Emulator 4', emulator_4_patch, 'emulator_4')
    ]
    
    depths = np.arange(n_depths)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(emulator_1_patch.time.values[t])
        
        llc_data = llc_patch.isel(time=t)[var].values  # (k, j, i)
        llc_grad_mask = gradient_masks['llc'][var][t]   # (k, j, i)
        
        # First pass: compute all errors for shared xlim
        row_mean_errors = []
        row_median_errors = []
        row_hg_mean_errors = []
        
        for emulator_name, emulator_patch, emulator_key in emulator_patches_list:
            emu_data = emulator_patch.isel(time=t)[var].values  # (k, j, i)
            diff = np.abs(llc_data - emu_data)  # (k, j, i)
            
            # All-pixel mean and median
            diff_flat = diff.reshape(n_depths, -1)
            mean_errors = np.nanmean(diff_flat, axis=1)
            median_errors = np.nanmedian(diff_flat, axis=1)
            
            # High-gradient mean error (using LLC mask)
            hg_mean_errors = np.zeros(n_depths)
            for k in range(n_depths):
                hg_pixels = diff[k][llc_grad_mask[k]]
                if len(hg_pixels) > 0:
                    hg_mean_errors[k] = np.nanmean(hg_pixels)
                else:
                    hg_mean_errors[k] = np.nan
            
            row_mean_errors.append(mean_errors)
            row_median_errors.append(median_errors)
            row_hg_mean_errors.append(hg_mean_errors)
        
        # Shared x-axis limits for this row
        all_errors = np.concatenate(row_mean_errors + row_median_errors + row_hg_mean_errors)
        xmin = 0
        xmax = np.nanmax(all_errors) * 1.05
        
        # Second pass: plot
        for col, (emulator_name, _, _) in enumerate(emulator_patches_list):
            ax = axes[row, col]
            
            mean_errors = row_mean_errors[col]
            median_errors = row_median_errors[col]
            hg_mean_errors = row_hg_mean_errors[col]
            
            # Mean (blue)
            ax.scatter(mean_errors, depths, color='blue', s=30, alpha=0.7, zorder=3)
            ax.plot(mean_errors, depths, color='blue', alpha=0.4, linewidth=1.5, label='Mean')
            
            # Median (red)
            ax.scatter(median_errors, depths, color='red', s=30, alpha=0.7, zorder=3)
            ax.plot(median_errors, depths, color='red', alpha=0.4, linewidth=1.5, label='Median')
            
            # High-gradient mean (green)
            ax.scatter(hg_mean_errors, depths, color='green', s=30, alpha=0.7, zorder=3)
            ax.plot(hg_mean_errors, depths, color='green', alpha=0.4, linewidth=1.5, label='HG Mean')
            
            # Variable-specific reference lines
            for ref_val in ref_lines[var]:
                ax.axvline(x=ref_val, color='black', linestyle='--', linewidth=1.5, alpha=0.5, zorder=2)
            
            ax.set_title(f'{emulator_name} {var} {time_str}', fontsize=8)
            ax.set_xlabel('Abs Error', fontsize=7)
            ax.set_ylabel('Depth (k)', fontsize=7)
            ax.set_ylim(n_depths - 1, 0)
            ax.set_xlim(xmin, xmax)
            ax.grid(alpha=0.2)
            ax.tick_params(labelsize=6)
            
            if col == 0:
                ax.legend(fontsize=6, loc='lower right')
    
    plt.tight_layout()
    plt.savefig(f'figs/prognostic_var_comparison/{var}/depth_error_by_time.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved augmented depth error plots for {var}")

print("Done!")

Generating augmented depth error plots for Theta...
✓ Saved augmented depth error plots for Theta
Generating augmented depth error plots for Salt...
✓ Saved augmented depth error plots for Salt
Generating augmented depth error plots for U...
✓ Saved augmented depth error plots for U
Generating augmented depth error plots for V...
✓ Saved augmented depth error plots for V
Done!
